# Проект 1. Классификация ирисов методом k ближайших соседей

Нужно по четырём измерениям цветка (длина и ширина чашелистика и лепестка) определить, к какому из трёх видов он относится: setosa, versicolor или virginica.

Данные: классический набор Iris (Фишер, 1936), 150 цветков, по 50 каждого вида. Беру его из пакета `RDatasets`.

Метод KNN: для цветка из теста находим 5 ближайших цветков из обучающей выборки (по евклидову расстоянию) и выбираем вид, который среди них встречается чаще. Для быстрого поиска соседей использую KD-дерево из `NearestNeighbors`.

Данные делю на обучающую и тестовую выборки 80/20, качество оцениваю по точности (accuracy) на тесте.

In [ ]:
import Pkg
Pkg.add(["RDatasets", "DataFrames", "NearestNeighbors", "MLBase", "Random",])

In [3]:
using RDatasets, DataFrames, NearestNeighbors, MLBase, Random

# Загружаем данные
iris = dataset("datasets", "iris")
X = Matrix(iris[:, 1:4])
y = iris[:, 5]

# Перемешиваем и делим 80/20
Random.seed!(42)
n = size(X, 1)
idx = shuffle(1:n)
train_idx = idx[1:Int(0.8*n)]
test_idx  = idx[Int(0.8*n)+1:end]

X_train, y_train = X[train_idx, :], y[train_idx]
X_test,  y_test  = X[test_idx, :],  y[test_idx]

# Строим KD-дерево по обучающим данным
tree = KDTree(permutedims(X_train))

# Для каждой точки теста находим k=5 соседей и голосуем
k = 5
y_pred = Vector{String}(undef, size(X_test, 1))
for i in 1:size(X_test, 1)
    idxs, _ = knn(tree, X_test[i, :], k)
    labels = y_train[idxs]
    # большинство голосов
    y_pred[i] = first(sort(unique(labels), by = l -> -count(==(l), labels)))
end

# Точность
accuracy = mean(y_pred .== y_test)
println("Точность KNN (k=$k): ", round(accuracy * 100, digits=2), "%")

Точность KNN (k=5): 100.0%


## Вывод

С k = 5 модель правильно определила вид у всех 30 цветков тестовой выборки, точность 100%.

Но это не значит, что модель никогда не ошибается. Тестовая выборка маленькая, и результат зависит от того, какие цветки в неё попали. При другом перемешивании (до того как я зафиксировал seed) точность была 96.7%, то есть одна ошибка из 30. Ошибаться KNN может только на versicolor и virginica: они похожи по размерам, и их области частично пересекаются. Setosa заметно меньше остальных и отделяется без проблем.

У KNN нет этапа обучения как такового: модель просто хранит обучающие точки и для нового цветка ищет ближайшие. Для Iris это работает хорошо, потому что данных мало и признаков всего четыре. На больших наборах поиск соседей был бы медленнее.